# Notebook 6: Image-to-Text (Comprensión Visual Profunda)

**Autores:** Javier Arroyo | Julia Cano | Paula Durá  
**Asignatura:** Procesamiento de Imágenes  

---

## Objetivo

Extraer **información semántica rica y estructurada** de las imágenes, yendo más allá de la descripción simple del Image Captioning (Notebook 05).

### Image Captioning vs Image-to-Text

| Image Captioning (Nb. 05) | Image-to-Text (este notebook) |
|---|---|
| *"a dog on a beach"* | Escena: playa. Sujeto: perro (golden retriever). Ambiente: soleado, exterior. |
| Una frase descriptiva | Análisis multi-dimensional estructurado |
| Solo describe | Responde preguntas, extrae atributos, genera texto controlado |

### Tareas implementadas

| # | Tarea | Modelo | Salida |
|---|-------|--------|--------|
| 1 | Visual Question Answering (VQA) | BLIP-VQA (preentrenado) | Respuestas a preguntas específicas |
| 2 | Extracción de atributos | BLIP-VQA | Fichas descriptivas estructuradas |
| 3 | Captions condicionados | BLIP (preentrenado) | Descripciones guiadas por prompt |
| 4 | Predicción de atributos | CNN multi-tarea (from scratch) | Texto desde clasificadores |

> **Nota:** Se usa el dataset aumentado del Notebook 01.

---
## 6.1 Configuración

Importamos `transformers` para BLIP-VQA y TensorFlow/Keras para el modelo from scratch.

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
import random
import warnings
import json
warnings.filterwarnings("ignore")

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torchvision import transforms, models

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    BlipForQuestionAnswering
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# Cargar dataset
DATA_DIR = Path("dataset_augmented")
if not DATA_DIR.exists():
    DATA_DIR = Path("dataset")

classes = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
print("Clases:", classes)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
rows = []
for cls in classes:
    for p in (DATA_DIR / cls).rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            rows.append({"path": str(p), "class": cls})

df = pd.DataFrame(rows)
print(f"Total imágenes: {len(df)}")

# Muestra representativa: 8 por clase
sample_df = df.groupby("class").apply(lambda x: x.sample(min(8, len(x)), random_state=SEED)).reset_index(drop=True)
print(f"Muestra para análisis: {len(sample_df)}")

---
## 6.2 Visual Question Answering (VQA) con BLIP

### Concepto

**VQA** representa una forma de Image-to-Text **interactiva**: el usuario formula preguntas en lenguaje natural sobre una imagen y el modelo genera respuestas. Esto requiere:
- Comprensión visual (reconocer objetos, escenas, colores, acciones)
- Comprensión lingüística (entender la pregunta)
- Razonamiento multi-modal (conectar ambas)

Usamos **BLIP-VQA** (`Salesforce/blip-vqa-base`), un modelo especializado en esta tarea con ~361M de parámetros.

In [ ]:
# Cargar BLIP para VQA
vqa_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
vqa_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base").to(device)
vqa_model.eval()

print("BLIP-VQA model loaded successfully")
print(f"Parámetros: {sum(p.numel() for p in vqa_model.parameters()) / 1e6:.1f}M")

In [ ]:
def ask_image(img_path, question):
    """Hace una pregunta sobre una imagen y devuelve la respuesta."""
    img = Image.open(img_path).convert("RGB")
    inputs = vqa_processor(images=img, text=question, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output = vqa_model.generate(**inputs, max_length=30)
    
    answer = vqa_processor.decode(output[0], skip_special_tokens=True)
    return answer

# Test rápido
test_path = sample_df["path"].iloc[0]
test_class = sample_df["class"].iloc[0]
print(f"Imagen: {Path(test_path).name} (clase: {test_class})")
print(f"Q: What is in this image?")
print(f"A: {ask_image(test_path, 'What is in this image?')}")
print(f"Q: What colors do you see?")
print(f"A: {ask_image(test_path, 'What colors do you see?')}")

### 6.2.1 Batería de preguntas por categoría

Diseñamos un conjunto de preguntas **generales** (aplicables a toda imagen) y **específicas** (adaptadas a cada categoría) para evaluar la capacidad de comprensión del modelo.

In [ ]:
# Preguntas generales aplicables a todas las categorías
GENERAL_QUESTIONS = [
    "What is in this image?",
    "What colors are dominant?",
    "Is this indoors or outdoors?",
    "What is the mood or atmosphere?",
    "How many objects are visible?"
]

# Preguntas específicas por categoría
CATEGORY_QUESTIONS = {
    "Animales": [
        "What animal is this?",
        "What is the animal doing?",
        "Is the animal wild or domestic?"
    ],
    "Ciudad": [
        "What buildings can you see?",
        "Are there people in the image?",
        "Is this a modern or old city?"
    ],
    "Comida": [
        "What food is shown?",
        "Is this food cooked or raw?",
        "What cuisine is this?"
    ],
    "Naturaleza": [
        "What landscape is shown?",
        "Are there trees or mountains?",
        "What season does it look like?"
    ],
    "Playa": [
        "Is there water in the image?",
        "Are there people on the beach?",
        "What time of day is it?"
    ]
}

print("Preguntas por categoría definidas")
for cls, qs in CATEGORY_QUESTIONS.items():
    print(f"  {cls}: {len(GENERAL_QUESTIONS) + len(qs)} preguntas")

In [ ]:
# Ejecutar VQA sobre toda la muestra
vqa_results = []

for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="VQA inference"):
    img_path = row["path"]
    cls = row["class"]
    
    result = {"path": img_path, "class": cls}
    
    # Preguntas generales
    for q in GENERAL_QUESTIONS:
        try:
            a = ask_image(img_path, q)
        except:
            a = "[Error]"
        result[q] = a
    
    # Preguntas específicas de la categoría
    specific_qs = CATEGORY_QUESTIONS.get(cls, [])
    for q in specific_qs:
        try:
            a = ask_image(img_path, q)
        except:
            a = "[Error]"
        result[q] = a
    
    vqa_results.append(result)

vqa_df = pd.DataFrame(vqa_results)
print(f"VQA completado: {len(vqa_df)} imágenes analizadas")

In [ ]:
# Visualizar respuestas VQA (1 imagen por clase con preguntas generales)
fig, axes = plt.subplots(len(classes), 1, figsize=(14, 5 * len(classes)))

for i, cls in enumerate(classes):
    row = vqa_df[vqa_df["class"] == cls].iloc[0]
    img = Image.open(row["path"]).convert("RGB")
    axes[i].imshow(img)
    
    # Formar el texto de Q&A
    qa_text = f"[{cls}]\n"
    for q in GENERAL_QUESTIONS:
        if q in row:
            qa_text += f"  Q: {q}\n  A: {row[q]}\n"
    
    axes[i].set_title(qa_text, fontsize=9, loc="left", family="monospace")
    axes[i].axis("off")

plt.suptitle("Visual Question Answering (BLIP-VQA) por categoría", fontsize=14)
plt.tight_layout()
plt.show()

### 6.2.2 Análisis agregado de respuestas

Analizamos patrones en las respuestas VQA para verificar que el modelo captura correctamente las diferencias entre categorías (interior/exterior, colores dominantes, etc.).

In [ ]:
# Analizar respuestas "indoors or outdoors" por categoría
indoor_outdoor = vqa_df.groupby("class")["Is this indoors or outdoors?"].value_counts().unstack(fill_value=0)
print("Indoor/Outdoor por categoría:")
display(indoor_outdoor)

# Analizar colores dominantes
color_freq_by_class = {}
for cls in classes:
    cls_data = vqa_df[vqa_df["class"] == cls]
    colors = []
    for _, row in cls_data.iterrows():
        answer = str(row.get("What colors are dominant?", ""))
        # Extraer palabras de color
        color_words = ["red", "blue", "green", "yellow", "white", "black", "brown", 
                       "orange", "pink", "purple", "gray", "grey", "gold", "silver"]
        for cw in color_words:
            if cw in answer.lower():
                colors.append(cw)
    color_freq_by_class[cls] = Counter(colors)

# Visualizar colores dominantes
fig, axes = plt.subplots(1, len(classes), figsize=(5 * len(classes), 5))

for i, cls in enumerate(classes):
    freq = color_freq_by_class[cls]
    if freq:
        colors_list, counts = zip(*freq.most_common(6))
        # Mapear nombres a colores matplotlib
        color_map = {"red": "red", "blue": "blue", "green": "green", "yellow": "gold",
                     "white": "lightgray", "black": "black", "brown": "saddlebrown",
                     "orange": "orange", "pink": "pink", "purple": "purple",
                     "gray": "gray", "grey": "gray", "gold": "gold", "silver": "silver"}
        bar_colors = [color_map.get(c, "#3498db") for c in colors_list]
        axes[i].barh(range(len(colors_list)), counts, color=bar_colors)
        axes[i].set_yticks(range(len(colors_list)))
        axes[i].set_yticklabels(colors_list)
        axes[i].invert_yaxis()
    axes[i].set_title(f"{cls}", fontsize=12)
    axes[i].set_xlabel("Frecuencia")

plt.suptitle("Colores dominantes detectados por VQA por categoría", fontsize=14)
plt.tight_layout()
plt.show()

---
## 6.3 Extracción de atributos estructurados

### De imagen a ficha descriptiva

Usando VQA como herramienta, convertimos cada imagen en un **registro estructurado** con atributos predefinidos. Este enfoque permite:
- Construir **bases de datos** a partir de colecciones de imágenes
- Generar **metadatos** automáticos para sistemas de búsqueda
- Crear **descripciones estandarizadas** para catálogos

In [ ]:
# Preguntas para extracción de atributos
ATTRIBUTE_QUESTIONS = {
    "scene_type": "What type of scene is this?",
    "main_subject": "What is the main subject?",
    "time_of_day": "What time of day is it?",
    "weather": "What is the weather like?",
    "dominant_color": "What is the dominant color?",
    "environment": "Is this natural or man-made?",
    "people_present": "Are there people in this image?",
    "mood": "What mood does this image convey?"
}

# Extraer atributos para todas las imágenes
attributes = []

for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Extracting attributes"):
    attr = {"path": row["path"], "class": row["class"]}
    
    for attr_name, question in ATTRIBUTE_QUESTIONS.items():
        try:
            attr[attr_name] = ask_image(row["path"], question)
        except:
            attr[attr_name] = "unknown"
    
    attributes.append(attr)

attr_df = pd.DataFrame(attributes)
print("Atributos extraídos:")
display(attr_df[["class"] + list(ATTRIBUTE_QUESTIONS.keys())].head(10))

In [ ]:
# Mostrar fichas descriptivas por categoría
print("=" * 100)
print("FICHAS DESCRIPTIVAS GENERADAS POR IMAGE-TO-TEXT")
print("=" * 100)

for cls in classes:
    cls_data = attr_df[attr_df["class"] == cls].head(2)
    print(f"\n{'━'*80}")
    print(f"  Categoría: {cls.upper()}")
    print(f"{'━'*80}")
    for _, row in cls_data.iterrows():
        print(f"  ┌─ Ficha descriptiva ─────────────────────────────")
        for attr_name in ATTRIBUTE_QUESTIONS.keys():
            print(f"  │  {attr_name:20s}: {row[attr_name]}")
        print(f"  └──────────────────────────────────────────────────")
        print()

---
## 6.4 Generación de descripciones condicionadas

### Concepto

A diferencia del captioning estándar (Notebook 05), aquí usamos **prompts condicionales** para dirigir la generación de texto. Esto permite obtener descripciones enfocadas en diferentes aspectos de la imagen según las necesidades del usuario.

In [ ]:
# Cargar BLIP captioning (conditonal)
caption_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
caption_model_blip = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
caption_model_blip.eval()

def generate_conditional_caption(img_path, prompt, max_length=60):
    """Genera un caption condicionado por un prompt."""
    img = Image.open(img_path).convert("RGB")
    inputs = caption_processor(images=img, text=prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output = caption_model_blip.generate(**inputs, max_length=max_length, num_beams=5)
    
    caption = caption_processor.decode(output[0], skip_special_tokens=True)
    return caption

# Definir prompts condicionales
CONDITIONAL_PROMPTS = [
    "a photograph of",
    "this image shows",
    "the scene depicts",
    "in this picture there is",
    "a detailed description:"
]

print("Prompts condicionales definidos:", len(CONDITIONAL_PROMPTS))

In [ ]:
# Generar descripciones condicionadas para 1 imagen por clase
fig, axes = plt.subplots(len(classes), 1, figsize=(16, 6 * len(classes)))

for i, cls in enumerate(classes):
    img_path = sample_df[sample_df["class"] == cls]["path"].iloc[0]
    img = Image.open(img_path).convert("RGB")
    axes[i].imshow(img)
    
    caption_text = f"[{cls}]\n"
    for prompt in CONDITIONAL_PROMPTS:
        try:
            caption = generate_conditional_caption(img_path, prompt)
            caption_text += f'  Prompt: "{prompt}"\n  → {caption}\n\n'
        except:
            caption_text += f'  Prompt: "{prompt}"\n  → [Error]\n\n'
    
    axes[i].set_title(caption_text, fontsize=9, loc="left", family="monospace")
    axes[i].axis("off")

plt.suptitle("Descripciones condicionadas por prompt (BLIP)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 6.5 Modelo From Scratch: CNN + Clasificador Multi-tarea

### Enfoque

Construimos un pipeline **Image-to-Text desde cero** en tres pasos:

1. **Feature extraction:** ResNet18 preentrenada (congelada) → vector $\mathbf{v} \in \mathbb{R}^{512}$
2. **Clasificación multi-tarea:** Red densa que predice simultáneamente:
   - Categoría de escena (5 clases)
   - Indoor/Outdoor (binario)
   - Natural/Man-made (binario)
3. **Generación de texto:** Las predicciones se formatean en una descripción textual estructurada

### Arquitectura multi-tarea

```
ResNet18(frozen) → [512] → Dense(256) → Dense(128) → ┬─ Softmax(5) [escena]
                                                       ├─ Sigmoid(1) [outdoor]
                                                       └─ Sigmoid(1) [natural]
```

El entrenamiento multi-tarea permite que las capas compartidas aprendan representaciones más ricas, beneficiándose de la señal de supervisión múltiple.

In [ ]:
# Extraer features con ResNet18
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet_encoder = nn.Sequential(*list(resnet.children())[:-1])
resnet_encoder.eval()
resnet_encoder.to(device)

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_features(img_path):
    img = Image.open(img_path).convert("RGB")
    img_tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        features = resnet_encoder(img_tensor).squeeze()
    return features.cpu().numpy()

print("ResNet18 encoder loaded")

In [ ]:
# Preparar datos de entrenamiento con pseudo-labels de VQA
# Usamos más imágenes para entrenar el clasificador
train_sample = df.groupby("class").apply(lambda x: x.sample(min(40, len(x)), random_state=SEED)).reset_index(drop=True)

# Extraer features y pseudo-labels
print(f"Extrayendo features y pseudo-labels para {len(train_sample)} imágenes...")

features_list = []
labels_scene = []     # Nuestra categoría (5 clases)
labels_outdoor = []   # Indoor/Outdoor (binario)
labels_natural = []   # Natural/Man-made (binario)

class_to_idx = {c: i for i, c in enumerate(classes)}

for idx, row in tqdm(train_sample.iterrows(), total=len(train_sample), desc="Preparing data"):
    try:
        feat = extract_features(row["path"])
        features_list.append(feat)
        labels_scene.append(class_to_idx[row["class"]])
        
        # Pseudo-labels basadas en la categoría (heurísticas razonables)
        if row["class"] in ["Naturaleza", "Playa", "Animales"]:
            labels_outdoor.append(1)  # outdoor
            labels_natural.append(1 if row["class"] != "Animales" else 0)
        elif row["class"] == "Ciudad":
            labels_outdoor.append(1)  # outdoor
            labels_natural.append(0)  # man-made
        else:  # Comida
            labels_outdoor.append(0)  # indoor
            labels_natural.append(0)  # man-made
    except:
        pass

X_train_feat = np.array(features_list)
y_scene = np.array(labels_scene)
y_outdoor = np.array(labels_outdoor)
y_natural = np.array(labels_natural)

print(f"Features: {X_train_feat.shape}")
print(f"Scene labels: {Counter(y_scene)}")
print(f"Outdoor labels: {Counter(y_outdoor)}")

In [ ]:
from sklearn.model_selection import train_test_split

# Split
train_idx, val_idx = train_test_split(np.arange(len(X_train_feat)), test_size=0.2, random_state=SEED)

# Modelo multi-tarea: predice escena + outdoor + natural
feat_input = keras.Input(shape=(512,))

# Shared layers
x = layers.Dense(256, activation="relu")(feat_input)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)

# Cabeza 1: Clasificación de escena (5 clases)
scene_out = layers.Dense(len(classes), activation="softmax", name="scene")(x)

# Cabeza 2: Indoor/Outdoor (binario)
outdoor_out = layers.Dense(1, activation="sigmoid", name="outdoor")(x)

# Cabeza 3: Natural/Man-made (binario)
natural_out = layers.Dense(1, activation="sigmoid", name="natural")(x)

attr_model = keras.Model(feat_input, [scene_out, outdoor_out, natural_out], name="attribute_predictor")

attr_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss={
        "scene": "sparse_categorical_crossentropy",
        "outdoor": "binary_crossentropy",
        "natural": "binary_crossentropy"
    },
    loss_weights={"scene": 1.0, "outdoor": 0.5, "natural": 0.5},
    metrics={
        "scene": "accuracy",
        "outdoor": "accuracy",
        "natural": "accuracy"
    }
)

attr_model.summary()

In [ ]:
# Entrenar
history_attr = attr_model.fit(
    X_train_feat[train_idx],
    {
        "scene": y_scene[train_idx],
        "outdoor": y_outdoor[train_idx],
        "natural": y_natural[train_idx]
    },
    validation_data=(
        X_train_feat[val_idx],
        {
            "scene": y_scene[val_idx],
            "outdoor": y_outdoor[val_idx],
            "natural": y_natural[val_idx]
        }
    ),
    epochs=50,
    batch_size=16,
    callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)],
    verbose=1
)

### 6.5.1 Curvas de entrenamiento

Monitorizamos la accuracy de cada tarea por separado para ver cuáles se aprenden más fácilmente.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

h = history_attr.history

# Scene accuracy
axes[0].plot(h["scene_accuracy"], label="train")
axes[0].plot(h["val_scene_accuracy"], label="val")
axes[0].set_title("Scene Classification Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

# Outdoor accuracy
axes[1].plot(h["outdoor_accuracy"], label="train")
axes[1].plot(h["val_outdoor_accuracy"], label="val")
axes[1].set_title("Indoor/Outdoor Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

# Natural accuracy
axes[2].plot(h["natural_accuracy"], label="train")
axes[2].plot(h["val_natural_accuracy"], label="val")
axes[2].set_title("Natural/Man-made Accuracy")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.suptitle("Attribute Predictor - Entrenamiento", fontsize=13)
plt.tight_layout()
plt.show()

### 6.5.2 Generación de texto desde atributos predichos

Convertimos las predicciones del clasificador en **texto descriptivo** usando una plantilla. Aunque el resultado es mecánico (no es lenguaje natural fluido), demuestra el principio del pipeline Image→Features→Text.

In [ ]:
def image_to_text_scratch(img_path, model, classes):
    """Convierte una imagen en texto estructurado usando el modelo from scratch."""
    feat = extract_features(img_path)
    feat = feat[np.newaxis, :]
    
    scene_pred, outdoor_pred, natural_pred = model.predict(feat, verbose=0)
    
    scene_idx = np.argmax(scene_pred[0])
    scene_name = classes[scene_idx]
    scene_conf = scene_pred[0][scene_idx]
    
    is_outdoor = "outdoor" if outdoor_pred[0][0] > 0.5 else "indoor"
    outdoor_conf = outdoor_pred[0][0] if outdoor_pred[0][0] > 0.5 else 1 - outdoor_pred[0][0]
    
    is_natural = "natural environment" if natural_pred[0][0] > 0.5 else "man-made environment"
    natural_conf = natural_pred[0][0] if natural_pred[0][0] > 0.5 else 1 - natural_pred[0][0]
    
    # Construir texto descriptivo
    text = (
        f"This image shows a {scene_name.lower()} scene (confidence: {scene_conf:.1%}). "
        f"The setting appears to be {is_outdoor} ({outdoor_conf:.1%}) "
        f"in a {is_natural} ({natural_conf:.1%})."
    )
    
    return text, {
        "scene": scene_name,
        "scene_conf": scene_conf,
        "setting": is_outdoor,
        "environment": is_natural
    }

# Generar texto para la muestra
scratch_texts = []
for idx, row in sample_df.iterrows():
    try:
        text, attrs = image_to_text_scratch(row["path"], attr_model, classes)
        scratch_texts.append({
            "path": row["path"],
            "class": row["class"],
            "generated_text": text,
            **attrs
        })
    except:
        scratch_texts.append({
            "path": row["path"],
            "class": row["class"],
            "generated_text": "[Error]"
        })

scratch_text_df = pd.DataFrame(scratch_texts)

In [ ]:
# Visualización: comparación VQA (preentrenado) vs from scratch
fig, axes = plt.subplots(len(classes), 1, figsize=(16, 5 * len(classes)))

for i, cls in enumerate(classes):
    img_path = sample_df[sample_df["class"] == cls]["path"].iloc[0]
    img = Image.open(img_path).convert("RGB")
    axes[i].imshow(img)
    
    # VQA
    vqa_row = vqa_df[vqa_df["path"] == img_path]
    vqa_text = ""
    if len(vqa_row) > 0:
        vqa_row = vqa_row.iloc[0]
        for q in GENERAL_QUESTIONS[:3]:
            if q in vqa_row:
                vqa_text += f"    Q: {q} → {vqa_row[q]}\n"
    
    # Scratch
    scratch_row = scratch_text_df[scratch_text_df["path"] == img_path]
    scratch_text = scratch_row.iloc[0]["generated_text"] if len(scratch_row) > 0 else "N/A"
    
    title = (
        f"[{cls}]\n"
        f"  ── BLIP-VQA (preentrenado) ──\n{vqa_text}"
        f"  ── From Scratch ──\n    {scratch_text}"
    )
    axes[i].set_title(title, fontsize=9, loc="left", family="monospace")
    axes[i].axis("off")

plt.suptitle("Image-to-Text: Preentrenado (VQA) vs From Scratch", fontsize=14)
plt.tight_layout()
plt.show()

---
## 6.6 Evaluación cuantitativa del modelo from scratch

Evaluamos la accuracy de cada cabeza del clasificador multi-tarea en el conjunto de validación.

In [ ]:
# Evaluar la precisión del clasificador de atributos en validación
val_preds = attr_model.predict(X_train_feat[val_idx], verbose=0)

from sklearn.metrics import accuracy_score, classification_report

# Scene classification
scene_pred_labels = np.argmax(val_preds[0], axis=1)
scene_acc = accuracy_score(y_scene[val_idx], scene_pred_labels)
print(f"Scene classification accuracy: {scene_acc:.4f}")
print(classification_report(y_scene[val_idx], scene_pred_labels, target_names=classes))

# Outdoor
outdoor_pred_labels = (val_preds[1].ravel() > 0.5).astype(int)
outdoor_acc = accuracy_score(y_outdoor[val_idx], outdoor_pred_labels)
print(f"Indoor/Outdoor accuracy: {outdoor_acc:.4f}")

# Natural
natural_pred_labels = (val_preds[2].ravel() > 0.5).astype(int)
natural_acc = accuracy_score(y_natural[val_idx], natural_pred_labels)
print(f"Natural/Man-made accuracy: {natural_acc:.4f}")

In [ ]:
# Gráfico resumen de accuracy
fig, ax = plt.subplots(figsize=(8, 5))

tasks = ["Scene\n(5 clases)", "Indoor/Outdoor\n(binario)", "Natural/Man-made\n(binario)"]
accs = [scene_acc, outdoor_acc, natural_acc]
colors = ["#3498db", "#2ecc71", "#e74c3c"]

bars = ax.bar(tasks, accs, color=colors, alpha=0.8, edgecolor="black")
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{acc:.1%}', ha='center', va='bottom', fontweight='bold')

ax.set_ylim(0, 1.15)
ax.set_ylabel("Accuracy")
ax.set_title("Attribute Predictor (from scratch) - Accuracy por tarea")
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="Random baseline")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6.7 Comparación final: Preentrenado vs From Scratch

Resumen lado a lado de las capacidades de cada enfoque para la tarea Image-to-Text.

In [ ]:
# Tabla resumen
comparison_data = {
    "Aspecto": [
        "Modelo",
        "Tipo de salida",
        "Requiere entrenamiento",
        "Calidad de texto",
        "Vocabulario",
        "Capacidad de preguntas",
        "Descripción condicionada",
        "Información extraída"
    ],
    "BLIP-VQA (Preentrenado)": [
        "BLIP-VQA (Salesforce)",
        "Respuestas en lenguaje natural",
        "No",
        "Alta (fluida, detallada)",
        "Ilimitado",
        "Sí (cualquier pregunta)",
        "Sí (con prompts)",
        "Rica y variada"
    ],
    "CNN + Clasificador (Scratch)": [
        "ResNet18 + Dense multi-tarea",
        "Texto generado desde atributos",
        "Sí (con pseudo-labels)",
        "Baja (templated)",
        "Fijo (atributos predefinidos)",
        "No",
        "No",
        "Limitada (3 atributos)"
    ]
}

comparison_table = pd.DataFrame(comparison_data)
display(comparison_table.set_index("Aspecto"))

---
## 6.8 Conclusiones

### BLIP-VQA (Preentrenado)
- **Comprensión profunda y flexible**: responde cualquier pregunta sobre contenido visual
- **Extracción de atributos versátil**: basta con formular la pregunta adecuada
- **Generación condicionada**: los prompts permiten controlar el enfoque de las descripciones
- **Sin entrenamiento**: funciona directamente (*zero-shot*) en nuestro dominio

### CNN Multi-tarea (From Scratch)
- **Información limitada**: solo predice 3 atributos predefinidos
- **Texto mecánico**: plantillas fijas, no lenguaje natural
- **Accuracy razonable**: las features de ResNet18 son informativas para clasificación
- **Valor educativo**: demuestra la arquitectura del pipeline Image→Features→Text

### Reflexión sobre Image-to-Text

Este notebook ha explorado la **progresión de complejidad** en la conversión de imágenes a texto:

| Nivel | Tarea | Complejidad |
|-------|-------|-------------|
| 1 | Clasificación → etiqueta | Baja |
| 2 | Captioning → frase descriptiva | Media |
| 3 | VQA → respuestas a preguntas | Alta |
| 4 | Extracción → ficha estructurada | Alta |

Los modelos preentrenados de visión-lenguaje (BLIP, ViT-GPT2) representan un avance cualitativo masivo, permitiendo interacción flexible con contenido visual sin necesidad de datos de entrenamiento específicos del dominio.